# 🎬 Reel Insights: Data Cleaning & Relational Schema Setup
### Notebook 01: Multi-file Relational Cleaning, Integrity Enforcement & SQLite Seeding
**Project:** Reel Insights – TMDB Movie Analytics  
**Author:** Senior Data Analyst & Python Engineer  

---
### Objectives:
1. Load all six provided CSV files (`movies`, `genres`, `movie_genres`, `cast`, `crew`, `movie_keywords`).
2. Audit data quality: identify missing values, exact duplicate records, and invalid types.
3. Handle missing values responsibly — **never assume $0 budget or revenue means the film was free**.
4. Remove duplicate rows and eliminate orphaned relational references.
5. Define normalized SQLite schema with Primary and Foreign Keys and seed `data/movies.db`.
6. Run a validation JOIN query to confirm relational integrity.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import pandas as pd
import numpy as np

from src.cleaning import load_all_raw_datasets, execute_full_cleaning_pipeline
from src.database import create_schema, load_cleaned_tables_to_db, execute_query, DEFAULT_DB_PATH

pd.set_option("display.max_columns", None)
print("Dependencies loaded successfully!")


## 1. Inspect Raw CSV Datasets
Let us inspect the dimensions, column types, and missing value patterns across all six raw files.


In [ ]:
raw_tables = load_all_raw_datasets(PROJECT_ROOT / "data")

for name, df in raw_tables.items():
    print(f"=== Table: {name} ===")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Duplicates: {df.duplicated().sum()}")
    nulls = df.isnull().sum()
    null_cols = nulls[nulls > 0]
    if not null_cols.empty:
        print(f"Missing Values:\n{null_cols}")
    else:
        print("Missing Values: None")
    print("-" * 50)


## 2. Audit Financial Placeholder Values in `movies.csv`
TMDB uses `0` to denote unavailable or unrecorded budget and revenue data. Let us verify how many movies contain these placeholders.


In [ ]:
movies_raw = raw_tables["movies"]
zero_bud = (movies_raw["budget"] == 0).sum()
zero_rev = (movies_raw["revenue"] == 0).sum()
both_zero = ((movies_raw["budget"] == 0) & (movies_raw["revenue"] == 0)).sum()
valid_both = ((movies_raw["budget"] > 0) & (movies_raw["revenue"] > 0)).sum()

print(f"Total Movies: {len(movies_raw):,}")
print(f"Budget == 0: {zero_bud} ({zero_bud / len(movies_raw) * 100:.1f}%)")
print(f"Revenue == 0: {zero_rev} ({zero_rev / len(movies_raw) * 100:.1f}%)")
print(f"Both Budget & Revenue == 0: {both_zero} ({both_zero / len(movies_raw) * 100:.1f}%)")
print(f"Valid Financial Records (>0): {valid_both} ({valid_both / len(movies_raw) * 100:.1f}%)")


> [!IMPORTANT]
> **Treatment Decision**: Budget and revenue values of $0$ will NOT be overwritten with arbitrary means or assumed as $0$ cost. In SQLite, they remain stored cleanly as their source values (or NULL where applicable), and in analytical pipelines, they are masked as `NaN` during ROI and profit margin calculations to prevent infinite or negative 100% false returns.


## 3. Execute End-to-End Cleaning & Deduplication Pipeline
We deduplicate primary entities and child junction records, strip text whitespace, parse release dates, and guarantee referential integrity.


In [ ]:
cleaned_tables = execute_full_cleaning_pipeline(PROJECT_ROOT / "data")

print("\nSample Cleaned Movies:")
display(cleaned_tables["movies"][["movie_id", "title", "release_date_str", "budget", "revenue", "popularity", "vote_average"]].head(5))


## 4. Initialize and Populate Relational SQLite Database
We create the SQLite database `data/movies.db` with enforced Primary Keys, Foreign Keys, and B-Tree indexes, and insert the cleaned records.


In [ ]:
db_file = PROJECT_ROOT / "data" / "movies.db"
print(f"Seeding SQLite database at: {db_file}")

load_cleaned_tables_to_db(cleaned_tables, db_file)
print("Database schema created and populated successfully!")


## 5. Sanity Verification Join Query
To confirm relational connectivity between `movies`, `movie_genres`, and `genres`, we execute a multi-table join.


In [ ]:
test_query = """
SELECT 
    m.movie_id,
    m.title,
    m.release_date,
    g.genre_name
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
ORDER BY m.release_date DESC
LIMIT 10;
"""
test_results = execute_query(test_query, db_path=db_file)
display(test_results)


### 💡 Cleaning Phase Insight & Recommendations
1. **Deduplication Impact:** Exactly 6 duplicate movie records and corresponding orphaned child records were pruned without any loss of unique film catalog data.
2. **Financial Integrity:** Preserving 0 as missing rather than substituting arbitrary medians guarantees that ROI distributions reflect authentic box-office performance.
3. **Database Readiness:** The relational database `data/movies.db` is now fully indexed and primed for performant SQL and Pandas analytics.
